In [1]:
import pandas as pd

# =========================
# FILE PATHS
# =========================
bom_file = r"D:/Tushar/main_with_subs_only.xlsx"
indent_file = r"D:/Inventory/Capacity Based 05-08 May'26.xlsx"

# =========================
# LOAD FILES
# =========================
bom_df = pd.read_excel(bom_file)
indent_df = pd.read_excel(indent_file)

bom_df.columns = bom_df.columns.str.strip()
indent_df.columns = indent_df.columns.str.strip()

def normalize(series):
    return series.astype(str).str.strip().str.upper()

# Normalize
bom_df['Sub_Label'] = normalize(bom_df['Sub_Label'])
bom_df['Main_Label'] = normalize(bom_df['Main_Label'])
indent_df['Material'] = normalize(indent_df['Material'])

# =========================
# BUILD LOOKUP MAPS
# =========================

# switch -> list of (child, usage)
bom_map = {}

for _, row in bom_df.iterrows():
    sub = row['Sub_Label']
    child = row['Main_Label']
    usage = pd.to_numeric(row['Sub_Count'], errors='coerce') or 0

    bom_map.setdefault(sub, []).append((child, usage))

children = bom_df['Main_Label'].unique()
output_df = pd.DataFrame({'Material': children})

# =========================
# PROCESS COLUMNS
# =========================

for col in indent_df.columns:

    if col == 'Material':
        continue

    print(f"Processing: {col}")

    indent_df['Demand'] = pd.to_numeric(indent_df[col], errors='coerce').fillna(0)

    results = {}

    for _, row in indent_df.iterrows():

        part = row['Material']
        demand = row['Demand']

        # Case 1 — switch part → explode via BOM
        if part in bom_map:
            for child, usage in bom_map[part]:
                results[child] = results.get(child, 0) + demand * usage

        # Case 2 — already child part → direct add
        elif part in children:
            results[part] = results.get(part, 0) + demand

        # Else ignore

    col_df = pd.DataFrame(list(results.items()), columns=['Material', col])

    output_df = output_df.merge(col_df, on='Material', how='left')

output_df = output_df.fillna(0)

# =========================
# SAVE
# =========================
output_df.to_excel("Child_for_10TH may_actual.xlsx", index=False)

# output_df.to_excel("Child part indent may 2026.xlsx", index=False)

print("✅ Saved output file.")


Processing: Segment
Processing: 2026-05-06 Total Production Plan
Processing: 2026-05-07 Total Production Plan
Processing: 2026-05-08 Total Production Plan
Processing: Unnamed: 5
Processing: Material.1
Processing: Sumif
Processing: Comparision
✅ Saved output file.
